# Naive model

Main idea: prediction on a particular day = average of all readings on that day before. 

If there are no readings on the same day before, do forward fill.


In [1]:
#import everything!
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import date, datetime, timedelta
from itertools import product
from copy import deepcopy
from PreRun import PreRun, PostRun
from sklearn.metrics import mean_absolute_error, mean_squared_error

#path to data
read_path = "../../../../data_ds_project/parquet_cleaned_energy"
#systems
good_systems_list = [4, 10, 33, 36, 50, 51, 1199, 1204, 1283, 1284, 1289, 1332, 4902, 4903]
reader_types = ["meter", "inverter", None]
#systems_cleaned
systems_cleaned = pd.read_csv("../../../data/core/systems_cleaned.csv")


In [2]:
# def naive_energy_forecaster(past_data: pd.DataFrame, times_to_predict: pd.DataFrame):
#     df = past_data.copy()
#     df['month_day'] = df['time'].dt.strftime('%m-%d')
#     df['time_of_day'] = df['time'].dt.time

#     avg_energy = (
#         df.groupby(['month_day', 'time_of_day'])['energy']
#         .mean()
#         .reset_index(name='energy_pred')
#     )

#     predictions = times_to_predict.copy()
#     predictions['month_day'] = predictions['time'].dt.strftime('%m-%d')
#     predictions['time_of_day'] = predictions['time'].dt.time
#     predictions = predictions.merge(
#         avg_energy,
#         on=['month_day', 'time_of_day'],
#         how='left'
#     )

#     predictions['energy_pred'] = predictions['energy_pred'].ffill().bfill()
    
#     return predictions['energy_pred']

In [3]:
# # experiment with system 4
# system_id=4
# check_prerun = PreRun(system_id=system_id, meter_or_inverter=None, path=read_path, systems_cleaned=systems_cleaned)
# check_prerun.fill_missing_hours()
# #do train test split
# #print("Good days:", check_prerun.good_days)
# good_days = check_prerun.good_days['date'].dt.date
# train_days = good_days[:int(0.8*len(good_days))]
# test_days = good_days[int(0.8*len(good_days)):]
# set_train_days = set(train_days)
# set_test_days = set(test_days)

# #print("check_prerun.data", check_prerun.data.head())

# train_data = check_prerun.data[check_prerun.data['time'].dt.date.isin(set_train_days)]
# test_data = check_prerun.data[check_prerun.data['time'].dt.date.isin(set_test_days)]

# # print("Train data:")
# # print(train_data.head())

# y_pred = naive_energy_forecaster(train_data, pd.DataFrame(test_data['time']))
# y_true = test_data['energy']

# #print(type(y_true.iloc[0]), type(y_pred.iloc[0]))



# print("custom error", PostRun.custom_error(y_true, y_pred, 1,2))


In [5]:
system_reader_pairs = [(4,None),(10,None), (33, None), (50,None), (51,None), (1283,'inverter'),(1283,'meter')]
# system_reader_pairs = [(1283,'inverter'),(1283,'meter')]
for pair in system_reader_pairs:
    system_id = pair[0]
    reader_type = pair[1]

    check_prerun = PreRun(system_id=system_id, meter_or_inverter=reader_type, path=read_path, systems_cleaned=systems_cleaned)
    check_prerun.fill_missing_hours()
    system_recorded_max = check_prerun.data['energy'].max()
    check_prerun.good_end_days_naive(1)
    check_prerun.tts_of_data_using_end_days()
    all_data = check_prerun.amended_data.copy()

    pred_days = (check_prerun.end_days_naive['date']).dt.date
    pred_days_set=set(pred_days)

    #make the predictions! will be made as a new column of all_data
    all_data['key'] = list(zip(all_data['time'].dt.month, all_data['time'].dt.day, all_data['time'].dt.hour))
    all_data['naive_pred'] = (
        all_data.groupby('key')['energy'] #group by same month/day/time
        .transform(lambda x: x.expanding().mean().shift(1)) #expanding average, then shift by one to not include this year in calculation
    )

    #make sure value between 0 and highest observed max
    all_data['naive_pred'] = np.clip(all_data['naive_pred'], 0, system_recorded_max)

    all_data['naive_pred'] = all_data['naive_pred'].ffill().bfill()

    predicted_times = all_data.loc[all_data['time'].dt.date.isin(pred_days_set)]

    diff = predicted_times['energy'] - predicted_times['naive_pred']
    predicted_times['error'] = np.where(
        diff > 0,
        1 * diff**2,
        2 * diff**2
    )
    predicted_times['date'] = predicted_times['time'].dt.date
    daily_error = predicted_times.groupby('date')['error'].mean()
    
    # y_true = all_data.loc[all_data['time'].dt.date.isin(pred_days_set)][['time','energy']]
    # y_pred = all_data.loc[all_data['time'].dt.date.isin(pred_days_set)][['time','naive_pred']]
    # diff = y_true-y_pred

    # errors = pd.Series(
    #     np.where(diff > 0, 1 * diff**2, 2 * diff**2),
    #      index=diff.index
    # )
    
        

    daily_error.to_csv(f'naive_errors/{system_id}_{reader_type}_naive_errors', index=False)
